# 01 Error and Input Diagnostics

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`Input_Sensitivity.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/Input_Sensitivity.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)
- [`Pixcel_Attribution.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/Pixcel_Attribution.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)
- [`plot_input_sensitivity.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/plot_input_sensitivity.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)
- [`Spatial_Response_Analysis.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/Spatial_Response_Analysis.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'05_model_optimize/MLP/Normal/scripts/Input_Sensitivity.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'e7efc72d71249f88f035e2744a0575d487c0281b'}, '05_model_optimize/MLP/Normal/scripts/Pixcel_Attribution.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'bb3056dd92abc2d40efa4e34c7554d344675f6f6'}, '05_model_optimize/MLP/Normal/scripts/plot_input_sensitivity.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '4cd6c5935e890900ca426012fbc6ec88ee43c734'}, '05_model_optimize/MLP/Normal/scripts/Spatial_Response_Analysis.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'f839894c1e1dae39b4f9cff4f8857d7707d3b61d'}}

## Input_Sensitivity

원본: [05_model_optimize/MLP/Normal/scripts/Input_Sensitivity.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/Input_Sensitivity.py)  
Git blob: `e7efc72d71249f88f035e2744a0575d487c0281b`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)

EXP유002 Normal MLP input sensitivity diagnosis (stage 3 ONLY).

Repeat unchanged Normal MLP baseline, validate predictions, then apply ONE
controlled image change at a time to all 19,200 official Validation and 2,560
current Handmade images. Official Test is neither loaded nor scored.
No fitting on Handmade; no model tuning; does not commit results to GitHub.

CLI 인자 (정적 추출): `--out`, `--batch-size`

In [ ]:
%%archive_source 05_model_optimize/MLP/Normal/scripts/Input_Sensitivity.py
#!/usr/bin/env python3
"""EXP유002 Normal MLP input sensitivity diagnosis (stage 3 ONLY).

Repeat unchanged Normal MLP baseline, validate predictions, then apply ONE
controlled image change at a time to all 19,200 official Validation and 2,560
current Handmade images. Official Test is neither loaded nor scored.
No fitting on Handmade; no model tuning; does not commit results to GitHub.
"""
from __future__ import annotations

import argparse
import csv
import gzip
import hashlib
import json
import platform
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import scipy
from scipy.ndimage import maximum_filter, minimum_filter, affine_transform
import sklearn
from sklearn.metrics import confusion_matrix, accuracy_score

HERE = Path(__file__).resolve().parent
ROOT = HERE.parents[3]
sys.path.insert(0, str(HERE))
from Pixcel_Attribution import (
    read_handmade, prior_official_cm, prior_handmade, official_indices,
    fixed_split, pipeline, build_baseline, LABELS, DEFAULT_DATA, diverse_selection,
)

@dataclass(frozen=True)
class Variant:
    name: str
    family: str
    value: object

VARIANTS = [Variant("original", "original", None)]
VARIANTS += [
    Variant(f"shift_{direction}{pixels}", "shift", (dx, dy))
    for pixels in (1, 2)
    for direction, dx, dy in (("L", -pixels, 0), ("R", pixels, 0),
                              ("U", 0, -pixels), ("D", 0, pixels))
]
VARIANTS += [Variant(f"brightness_{int(factor*100):03d}", "brightness", factor)
             for factor in (0.80, 0.90, 1.10, 1.20)]
VARIANTS += [Variant(f"stroke_{direction}_{int(frac*100):02d}", "stroke", (direction, frac))
             for direction in ("thin", "thick") for frac in (0.25, 0.40)]
VARIANTS += [Variant(f"scale_{int(factor*100):03d}", "scale", factor)
             for factor in (0.90, 0.95, 1.05, 1.10)]

PAIR_LIST = [(a, b) for a, b in [
    ("5","F"), ("4","F"), ("7","F"), ("9","F"),
    ("B","8"), ("8","B"), ("0","D"), ("D","0"),
    ("9","4"), ("4","9"), ("E","C"), ("3","5"), ("7","2"),
]]
REF_DIR = ROOT / "05_model_optimize" / "MLP" / "Normal" / "results" / "Pixcel_Attribution"


def transform(batch: np.ndarray, condition: Variant):
    a = np.asarray(batch, dtype=np.float32).reshape(-1, 28, 28)
    group = condition.family
    if group == "original":
        z = a.copy()
    elif group == "shift":
        dx, dy = condition.value
        z = np.zeros_like(a)
        ys = slice(max(0, -dy), min(28, 28-dy))
        yd = slice(max(0, dy), min(28, 28+dy))
        xs = slice(max(0, -dx), min(28, 28-dx))
        xd = slice(max(0, dx), min(28, 28+dx))
        z[:, yd, xd] = a[:, ys, xs]
    elif group == "brightness":
        z = np.clip(a * float(condition.value), 0, 255)
    elif group == "stroke":
        direction, fraction = condition.value
        if direction == "thick":
            modified = maximum_filter(a, size=(1, 3, 3), mode="constant", cval=0)
        else:
            modified = minimum_filter(a, size=(1, 3, 3), mode="constant", cval=0)
        z = (1-fraction)*a + fraction*modified
    elif group == "scale":
        inv = 1.0 / float(condition.value)
        offset = 13.5 * (1.0 - inv)
        z = affine_transform(a, matrix=np.diag([1, inv, inv]),
                             offset=(0.0, offset, offset),
                             output_shape=a.shape, order=1, mode="constant",
                             cval=0.0, prefilter=False)
    else:
        raise ValueError(f"Unknown transformation: {condition}")
    return np.rint(np.clip(z, 0, 255)).astype(np.float32).reshape(len(a), 784)


def test_transform():
    x = np.zeros((2, 784), dtype=np.float32)
    x[0, 13*28+14] = 255
    x[1, 5*28+4] = 180
    assert len(VARIANTS) == 21 and len({v.name for v in VARIANTS}) == 21
    for v in VARIANTS:
        out = transform(x, v)
        assert out.shape == x.shape and out.dtype == np.float32
        assert out.min() >= 0 and out.max() <= 255
    # Spatial shift must never wrap border values.
    x2 = np.zeros((1, 784), dtype=np.float32)
    x2[0, 0] = 255
    assert not transform(x2, Variant("L1", "shift", (-1, 0))).any()
    assert np.array_equal(transform(x, VARIANTS[0]), x)
    print("TRANSFORM SELF-TEST PASSED (21 original/perturbed conditions)", flush=True)


def get_output(estimator, raw):
    """Return class predictions, softmax probabilities, true logit margin input.

    Internal one-hidden-layer ReLU forward pass; checked against sklearn
    Pipeline.predict_proba on original input before any sensitivity analysis.
    """
    net = estimator.named_steps["model"]
    if len(net.coefs_) != 2 or net.activation != "relu":
        raise RuntimeError("Baseline MLP architecture changed")
    x01 = np.asarray(raw, dtype=np.float32) / np.float32(255.0)
    hidden = np.maximum(x01 @ net.coefs_[0] + net.intercepts_[0], 0)
    logits = hidden @ net.coefs_[1] + net.intercepts_[1]
    z = logits.astype(np.float64)
    z -= z.max(axis=1, keepdims=True)
    probability = np.exp(z)
    probability /= probability.sum(axis=1, keepdims=True)
    return logits, probability


def margin(logits, y):
    ids = np.arange(len(y))
    true_score = logits[ids, y].astype(np.float64)
    competing = logits.copy()
    competing[ids, y] = -np.inf
    return true_score - competing.max(axis=1)


def record_summary(domain, variant, y, oldpred, pred, pold, pnew,
                   marginold, marginnew, mask, group_name, group_value,
                   clipping=None, mass_delta=None):
    ix = np.flatnonzero(mask)
    yy, old, new = y[ix], oldpred[ix], pred[ix]
    correct_before = (old == yy)
    correct_after = (new == yy)
    orig_correct = int(correct_before.sum())
    orig_wrong = len(ix) - orig_correct
    corrections = int((~correct_before & correct_after).sum())
    induced = int((correct_before & ~correct_after).sum())
    flips = int((old != new).sum())
    avgclip = float(np.mean(clipping[ix])) if clipping is not None and len(ix) else 0.
    avgmass = float(np.mean(mass_delta[ix])) if mass_delta is not None and len(ix) else 0.
    return {
        "domain":domain, "variant":variant.name, "family":variant.family,
        "group_type":group_name, "group":group_value, "n":len(ix),
        "original_correct":orig_correct, "original_wrong":orig_wrong,
        "original_accuracy":orig_correct/len(ix), "accuracy":int(correct_after.sum())/len(ix),
        "accuracy_delta_pp":100*(int(correct_after.sum())-orig_correct)/len(ix),
        "flip_count":flips, "flip_rate":flips/len(ix),
        "corrections":corrections,
        "correction_rate_among_original_errors":corrections/orig_wrong if orig_wrong else None,
        "induced_errors":induced,
        "induction_rate_among_original_correct":induced/orig_correct if orig_correct else None,
        "other_wrong_to_wrong_changes":int((~correct_before & ~correct_after & (old != new)).sum()),
        "mean_true_probability_delta":float((pnew[ix, yy] - pold[ix, yy]).mean()),
        "mean_true_logit_margin_delta":float((marginnew[ix]-marginold[ix]).mean()),
        "mean_fraction_brightness_clipped":avgclip,
        "mean_relative_ink_mass_delta":avgmass,
    }


def write_dict_rows(writer, domain, condition, y, writers, indices,
                    oldpred, pred, pold, pnew, marginold, marginnew,
                    clip, mass_delta):
    n = len(y)
    for i in range(n):
        yi = int(y[i])
        writer.writerow({
            "domain": domain, "writer": int(writers[i]), "sample_index": int(indices[i]),
            "true": LABELS[yi], "variant": condition.name, "family": condition.family,
            "original_prediction": LABELS[int(oldpred[i])],
            "transformed_prediction": LABELS[int(pred[i])],
            "original_correct": int(oldpred[i] == yi),
            "transformed_correct": int(pred[i] == yi),
            "original_p_true":float(pold[i, yi]),
            "transformed_p_true":float(pnew[i, yi]),
            "original_margin":float(marginold[i]),
            "transformed_margin":float(marginnew[i]),
            "delta_margin":float(marginnew[i]-marginold[i]),
            "brightness_clipped_pixel_fraction":float(clip[i]),
            "relative_ink_mass_change":float(mass_delta[i]),
        })


def pair_records(domain, condition, y, p0, p1):
    out = []
    for a, b in PAIR_LIST:
        ai, bi = LABELS.index(a), LABELS.index(b)
        old_pair = (y == ai) & (p0 == bi)
        new_pair = (y == ai) & (p1 == bi)
        count = int(old_pair.sum())
        corrected = int((old_pair & (p1 == ai)).sum())
        unchanged = int((old_pair & (p1 == bi)).sum())
        to_other = int((old_pair & (p1 != bi) & (p1 != ai)).sum())
        out.append({
            "domain":domain, "variant":condition.name, "family":condition.family,
            "actual":a, "previous_error":b, "original_pair_errors":count,
            "new_pair_errors":int(new_pair.sum()), "pair_count_delta":int(new_pair.sum())-count,
            "original_pair_corrected":corrected,
            "pair_correction_rate":corrected/count if count else None,
            "original_pair_still_same_error":unchanged,
            "original_pair_changed_to_other_error":to_other,
        })
    return out


def csv_out(path, rows):
    if not rows:
        return
    with path.open("w", newline="", encoding="utf-8-sig") as f:
        dw = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        dw.writeheader()
        dw.writerows(rows)


def select_focus(xva, yva, pva, xv_indices, xhm, yhm, phm, whm, ih):
    path = REF_DIR / "selected_cases.csv"
    focus = []
    official_map = {int(v): i for i, v in enumerate(xv_indices)}
    handmade_map = {(int(w), int(i)): k for k,(w,i) in enumerate(zip(whm, ih))}
    used = set()
    with path.open("r", newline="", encoding="utf-8-sig") as f:
        for r in csv.DictReader(f):
            if r["domain"] == "OfficialVal":
                k = official_map[int(r["index"])]
                domain = "OfficialVal"
                actual = yva[k]
                pred = pva[k]
                identifier = ("OfficialVal", int(r["index"]))
                original = xva[k]
                writer = -1
            elif r["domain"] == "Handmade":
                writer, idx = int(r["writer"]), int(r["index"])
                k = handmade_map[(writer, idx)]
                domain = "Handmade"
                actual = yhm[k]
                pred = phm[k]
                identifier = ("Handmade", writer, idx)
                original = xhm[k]
            else:
                raise RuntimeError(f"Unexpected prior selection domain {r['domain']}")
            if LABELS[int(actual)] != r["true"]:
                raise RuntimeError(f"Stage 1 selected image label changed {identifier}")
            prediction_match = (LABELS[int(pred)] == r["predicted"])
            if identifier in used:
                raise RuntimeError(f"Duplicate Stage 1 focus image {identifier}")
            focus.append({"domain":domain, "writer":writer,
                          "index":int(r["index"]), "true":int(actual),
                          "original_pred":int(pred), "pixels":original.copy(),
                          "case":r["case"], "origin":"stage1_2",
                          "stage12_pred":r["predicted"], "stage12_prediction_match":prediction_match})
            used.add(identifier)
    if len(focus) != 98:
        raise RuntimeError(f"Expected 98 Stage 1-2 cases, received {len(focus)}")
    for a, b in (("9", "F"), ("7", "F")):
        matches = np.flatnonzero((yhm==LABELS.index(a)) & (phm==LABELS.index(b)))
        extras = diverse_selection(matches, whm, 4, True)
        if len(extras) != 4:
            raise RuntimeError(f"Only {len(extras)} valid samples for {a}->{b}")
        for k in extras:
            w, idx = int(whm[k]), int(ih[k])
            identifier = ("Handmade", w, idx)
            if identifier in used:
                raise RuntimeError(f"Duplicate extra sample {identifier}")
            focus.append({"domain":"Handmade", "writer":w,"index":idx,
                          "true":int(yhm[k]),"original_pred":int(phm[k]),
                          "pixels":xhm[k].copy(),"case":f"{a}_to_{b}","origin":"stage3_extra",
                          "stage12_pred":LABELS[int(phm[k])], "stage12_prediction_match":True})
            used.add(identifier)
    if len(focus) != 106:
        raise RuntimeError("Expected exactly 106 focus images")
    return focus


def analyse_focus(focus, net, out):
    cols = []
    variant_names = {v.name:v for v in VARIANTS}
    for ex in focus:
        original = ex["pixels"].reshape(1,784)
        for variant in VARIANTS:
            changed = transform(original, variant)
            z, prob = get_output(net, changed)
            p = int(np.argmax(z, axis=1)[0])
            y = int(ex["true"])
            rival = z[0].copy()
            rival[y] = -np.inf
            score_margin = float(z[0,y] - rival.max())
            cols.append({
                "domain":ex["domain"], "writer":ex["writer"],"index":ex["index"],
                "true":LABELS[y],"original_prediction":LABELS[ex["original_pred"]],
                "case":ex["case"], "origin":ex["origin"],
                "stage12_pred":ex["stage12_pred"],
                "stage12_prediction_match":int(ex["stage12_prediction_match"]),
                "variant":variant.name, "family":variant.family,
                "new_prediction":LABELS[p], "new_correct":int(y==p),
                "p_true":float(prob[0,y]),
                "p_original_prediction":float(prob[0,ex["original_pred"]]),
                "true_logit_margin":score_margin,
            })
    csv_out(out/"stage3_focus_106.csv", cols)
    chosen = []
    for a,b in (("5","F"),("B","8"),("9","4"),("0","D"),("7","F")):
        ch = next((f for f in focus if f["domain"]=="Handmade" and
                   f["true"]==LABELS.index(a) and f["original_pred"]==LABELS.index(b)), None)
        if ch:
            chosen.append(ch)
    display_conditions = ("original", "shift_R2", "brightness_080", "stroke_thick_40", "scale_090")
    fig, axs = plt.subplots(len(chosen),len(display_conditions),
                            figsize=(12.6,2.65*len(chosen)), squeeze=False)
    for i, ex in enumerate(chosen):
        for j,name in enumerate(display_conditions):
            var=variant_names[name]
            x=transform(ex["pixels"].reshape(1,784),var)
            logits,probs=get_output(net,x)
            pred=int(logits[0].argmax())
            axs[i,j].imshow(x.reshape(28,28), cmap="gray",vmin=0,vmax=255)
            axs[i,j].set_title(f"{name}\n{LABELS[ex['true']]}->{LABELS[pred]} p(true)={probs[0,ex['true']]:.3f}",fontsize=8)
            axs[i,j].set_xticks([])
            axs[i,j].set_yticks([])
    fig.suptitle("Controlled one-variable perturbations (illustrative; not model optimization)",fontsize=12)
    fig.tight_layout()
    fig.savefig(out/"stage3_example_grid.png",dpi=145)
    plt.close(fig)
    fig, axs=plt.subplots(3,1,figsize=(13.3,9.2),squeeze=False)
    for i, ex in enumerate(chosen[:3]):
        ax=axs[i,0]
        rows=[r for r in cols if r["domain"]==ex["domain"] and
              r["writer"]==ex["writer"] and r["index"]==ex["index"]]
        ax.plot(range(len(rows)),[r["p_true"] for r in rows],marker="o",label="P(actual)")
        ax.plot(range(len(rows)),[r["p_original_prediction"] for r in rows],
                marker="x",label="P(original prediction)")
        ax.set_xticks(range(len(rows)),[r["variant"] for r in rows],rotation=65,ha="right",fontsize=7)
        ax.set_ylim(-0.05,1.05)
        ax.set_title(f"Writer {ex['writer']} image #{ex['index']}: "
                     f"{LABELS[ex['true']]} -> {LABELS[ex['original_pred']]}")
        ax.set_ylabel("Predicted probability")
        ax.legend(loc="best")
    fig.tight_layout()
    fig.savefig(out/"stage3_probability_curves.png",dpi=150)
    plt.close(fig)
    return cols


def run(args):
    start = time.perf_counter()
    test_transform()
    if args.batch_size < 64:
        raise ValueError("batch-size must be at least 64")
    out = args.out.resolve()
    out.mkdir(parents=True,exist_ok=True)
    xtr,ytr,xva,yva,offsha = fixed_split(DEFAULT_DATA)
    va_indices = official_indices(DEFAULT_DATA,yva)
    xhm,yhm,whm,ih = read_handmade()
    model_def,scaled = build_baseline()
    if not scaled or model_def.get_params()["random_state"] != 42:
        raise RuntimeError("Unexpected baseline hyperparameter change")
    model=pipeline(model_def,scaled)
    print("Re-fitting Normal MLP from official train only",flush=True)
    with warnings.catch_warnings(record=True) as ws:
        warnings.simplefilter("always")
        model.fit(xtr,ytr)
    pva=model.predict(xva).astype(int)
    phm=model.predict(xhm).astype(int)
    cm_l1=int(np.abs(confusion_matrix(yva,pva,labels=np.arange(16))
                   -prior_official_cm()).sum())
    hand_hist=prior_handmade()
    diffs=sum(hand_hist[(int(w),int(i))] != (LABELS[int(a)],LABELS[int(p)])
              for w,i,a,p in zip(whm,ih,yhm,phm))
    if cm_l1 != 0 or diffs != 0:
        # Fixed random_state and identical split do not imply identical fitted
        # floating-point weights across CPU/BLAS environments. Report mismatch
        # explicitly; compare each perturbation against THIS RUN'S fixed model.
        print(f"REPRODUCIBILITY WARNING: official_cm_L1={cm_l1}, "
              f"handmade_prediction_disagreements={diffs}. "
              "Stage 3 is an independent same-configuration refit, not the "
              "exact weights used for Stage 1-2.", flush=True)
    if len(yva)!=19200 or len(yhm)!=2560:
        raise RuntimeError("Expected source sizes changed")
    print(f"BASELINE CONFIRMED: official={accuracy_score(yva,pva):.8f}, "
          f"handmade={accuracy_score(yhm,phm):.8f}, cm_L1={cm_l1}, hand_pred_diff={diffs}",flush=True)
    # Drop training arrays to reduce peak runner RAM.
    del xtr,ytr
    focus = select_focus(xva,yva,pva,va_indices,xhm,yhm,phm,whm,ih)
    mismatch_focus = sum(not f["stage12_prediction_match"] for f in focus if f["origin"]=="stage1_2")
    print(f"FOCUS SAMPLE CHECK: {mismatch_focus}/98 have different predictions from Stage 1-2", flush=True)
    # Save current fitted model weights for independent audits of this single
    # perturbation experiment (never write it back into a model candidate).
    import joblib
    joblib.dump(model, out/"normal_mlp_refit_stage3.joblib", compress=3)
    deviations=[]
    for w,i,y,p in zip(whm,ih,yhm,phm):
        historical=hand_hist[(int(w),int(i))]
        if historical[1]!=LABELS[int(p)]:
            deviations.append({"writer":int(w),"image_index":int(i),
                               "true":LABELS[int(y)],
                               "stage12_prediction":historical[1],
                               "stage3_prediction":LABELS[int(p)]})
    csv_out(out/"stage3_baseline_prediction_differences.csv",deviations)
    focus_rows = analyse_focus(focus, model, out)
    focus_ids=len(focus)

    domains=[
      ("OfficialVal",xva,yva,np.full(len(yva),-1,dtype=np.int32),va_indices,pva),
      ("Handmade",xhm,yhm,whm,ih,phm)
    ]
    summaries, per_class, per_writer, pairs = [],[],[],[]
    fields=["domain","writer","sample_index","true","variant","family",
            "original_prediction","transformed_prediction","original_correct",
            "transformed_correct","original_p_true","transformed_p_true",
            "original_margin","transformed_margin","delta_margin",
            "brightness_clipped_pixel_fraction","relative_ink_mass_change"]
    total_rows=0
    with gzip.open(out/"stage3_predictions.csv.gz","wt",newline="",encoding="utf-8") as gz:
        writer=csv.DictWriter(gz,fieldnames=fields)
        writer.writeheader()
        for domain, x, y, authors, srcidx, oldpred in domains:
            base_logits, base_probs=get_output(model,x)
            pfromlogits=np.argmax(base_logits,axis=1)
            if not np.array_equal(pfromlogits,oldpred):
                raise RuntimeError(f"Internal forward pass is inconsistent for {domain}")
            test_proba=model.predict_proba(x[:25])
            if not np.allclose(base_probs[:25],test_proba,rtol=2e-5,atol=2e-6):
                raise RuntimeError("Internal softmax outputs deviate from Pipeline.predict_proba")
            m0=margin(base_logits,y)
            p0=base_probs
            for j, variant in enumerate(VARIANTS):
                n=len(y)
                preds=np.empty(n,dtype=np.int64)
                pn=np.empty((n,16),dtype=np.float64)
                mn=np.empty(n,dtype=np.float64)
                clip=np.zeros(n,dtype=np.float64)
                mass_delta=np.zeros(n,dtype=np.float64)
                for left in range(0,n,args.batch_size):
                    right=min(n,left+args.batch_size)
                    xb=x[left:right]
                    xt=transform(xb,variant)
                    logits,proba=get_output(model,xt)
                    pred=logits.argmax(axis=1)
                    preds[left:right]=pred
                    pn[left:right]=proba
                    mn[left:right]=margin(logits,y[left:right])
                    orig_sum=xb.sum(axis=1)
                    trans_sum=xt.sum(axis=1)
                    mass_delta[left:right]=(trans_sum-orig_sum)/np.maximum(orig_sum,1.0)
                    if variant.family=="brightness":
                        frac=float(variant.value)
                        clip[left:right]=((xb*frac)>255).sum(axis=1)/784.0
                if variant.family=="original" and (
                    not np.array_equal(preds,oldpred) or
                    not np.allclose(pn,p0,rtol=1e-7,atol=1e-7)):
                    raise RuntimeError(f"Identity condition altered baseline: {domain}")
                write_dict_rows(writer,domain,variant,y,authors,srcidx,
                                oldpred,preds,p0,pn,m0,mn,clip,mass_delta)
                total_rows+=n
                allmask=np.ones(n,dtype=bool)
                summaries.append(record_summary(domain,variant,y,oldpred,preds,p0,pn,
                                               m0,mn,allmask,"all","all",clip,mass_delta))
                for li, label in enumerate(LABELS):
                    per_class.append(record_summary(domain,variant,y,oldpred,preds,p0,pn,
                                                     m0,mn,y==li,"class",label,clip,mass_delta))
                if domain=="Handmade":
                    for w in (41,42,43,44):
                        per_writer.append(record_summary(domain,variant,y,oldpred,preds,p0,pn,
                                                        m0,mn,authors==w,"writer",str(w),clip,mass_delta))
                pairs.extend(pair_records(domain,variant,y,oldpred,preds))
                print(f"{domain:11s} {j+1:02d}/21 {variant.name:20s}: "
                      f"acc={summaries[-1]['accuracy']:.4f}, flip={summaries[-1]['flip_rate']:.4f}, "
                      f"corrected={summaries[-1]['corrections']}, induced={summaries[-1]['induced_errors']}",
                      flush=True)
    if total_rows != (19200+2560)*21:
        raise RuntimeError(f"Unexpected prediction total {total_rows}")
    csv_out(out/"stage3_sensitivity_summary.csv",summaries)
    csv_out(out/"stage3_per_class.csv",per_class)
    csv_out(out/"stage3_per_writer.csv",per_writer)
    csv_out(out/"stage3_error_pairs.csv",pairs)

    metadata={
        "scope":"Normal MLP Stage 3 input-only sensitivity experiment",
        "source_code":"05_model_optimize/MLP/Normal/scripts/Input_Sensitivity.py",
        "official_test_read":False,
        "handmade_training":False,
        "model_hyperparameters_modified":False,
        "data_sha256":offsha,
        "official_n":19200,"handmade_n":2560,
        "n_variants_including_original":len(VARIANTS),
        "n_nontrivial_variants":len(VARIANTS)-1,
        "total_prediction_rows":total_rows,
        "original_accuracy":{"OfficialVal":float(accuracy_score(yva,pva)),
                            "Handmade":float(accuracy_score(yhm,phm))},
        "baseline_validation_cm_L1_difference":cm_l1,
        "baseline_handmade_prediction_difference":diffs,
        "baseline_not_identical_to_stage12":bool(cm_l1!=0 or diffs!=0),
        "stage12_focus_changed_prediction_count":mismatch_focus,
        "focus_n":focus_ids, "focus_rows":len(focus_rows),
        "transforms":[{"name":v.name,"family":v.family,"value":v.value} for v in VARIANTS],
        "floating_precision":"mixed float32 fitted MLP forward, float64 softmax",
        "brightness_clip_fraction_basis":"all 784 image pixels",
        "ink_mass_change_basis":"relative sum of 0-255 pixel intensity",
        "warnings":[f"{w.category.__name__}: {w.message}" for w in ws],
        "elapsed_seconds":time.perf_counter()-start,
        "versions":{"python":platform.python_version(),"sklearn":sklearn.__version__,
                    "numpy":np.__version__,"scipy":scipy.__version__,
                    "matplotlib":matplotlib.__version__},
    }
    (out/"stage3_run_metadata.json").write_text(json.dumps(metadata,indent=2,ensure_ascii=False),encoding="utf-8")
    full=[r for r in summaries if r["group_type"]=="all" and r["variant"]!="original"]
    lines=[
      "# Normal MLP Input_Sensitivity — Stage 3 result",
      "",
      f"- Source: {metadata['official_n']:,} official Validation + {metadata['handmade_n']:,} Handmade.",
      f"- Original accuracy: official {metadata['original_accuracy']['OfficialVal']:.6%}; Handmade {metadata['original_accuracy']['Handmade']:.6%}.",
      f"- Historical reproduction check: official CM L1={cm_l1}; Handmade prediction discrepancies={diffs}.",
      f"- Prior 98 focus samples with changed predictions in this refit: {mismatch_focus}.",
      "- Model identity: same-data, same-hyperparameter independent refit; results are paired within THIS refit.",
      "- Any discrepancy from Stage 1-2 must not be presented as an identical-weight comparison.",
      f"- Controlled modifications: 8 shift, 4 brightness, 4 stroke, 4 scale = 20, plus original.",
      f"- Total scored rows: {total_rows:,}; focus examples: {focus_ids} (98 previous + 8 extra).",
      "- No official Test; no Handmade fitting, refit only once on Normal official Training.",
      "",
      "## Highest observed prediction flip conditions (each dataset)",
      "",
      "| Dataset | Transform | Flip rate | Accuracy delta(pp) | Corrected wrong | Newly wrong |",
      "|---|---|---:|---:|---:|---:|",
    ]
    for domain in ("OfficialVal","Handmade"):
        group=sorted((r for r in full if r["domain"]==domain),
                     key=lambda t:t["flip_rate"],reverse=True)
        for r in group[:5]:
            lines.append(
                f"| {domain} | {r['variant']} | {r['flip_rate']:.2%} | "
                f"{r['accuracy_delta_pp']:+.2f} | {r['corrections']} | {r['induced_errors']} |"
            )
    lines.extend([
       "",
       "## How to read these results",
       "",
       "- Flip rate: original predicted class differs from transformed prediction.",
       "- Correction rate: among originally incorrect inputs, fraction now correct.",
       "- Induction rate: among originally correct inputs, fraction now incorrect.",
       "- Accuracy delta: corrected count minus newly wrong count divided by all examples.",
       "- A high flip rate is sensitivity, not automatically an improvement.",
       "- Brightness clipping is measured explicitly. Thickness/scale transforms can alter discriminative strokes.",
       "- The comparison cannot by itself prove the model learned semantics rather than pixel correlations.",
       "- This is a diagnostic, not a selected preprocessing policy or a final performance gain.",
       "",
       "## Output index",
       "",
       "- stage3_predictions.csv.gz: ALL sample-level predictions (gzip-compressed CSV).",
       "- stage3_sensitivity_summary.csv: 21 conditions x 2 datasets.",
       "- stage3_per_class.csv: per-class metrics.",
       "- stage3_per_writer.csv: writer-level metrics.",
       "- stage3_error_pairs.csv: selected confusion-pair changes.",
       "- stage3_focus_106.csv: 106 original/focus images x 21 conditions.",
       "- stage3_example_grid.png and stage3_probability_curves.png: focused before/after views.",
       "- stage3_run_metadata.json: environment, conditions, reproducibility checks.",
       "- stage3_baseline_prediction_differences.csv: changed historical Handmade predictions, if any.",
       "- normal_mlp_refit_stage3.joblib: frozen weights for this run, in artifact only.",
    ])
    (out/"SUMMARY.md").write_text("\n".join(lines)+"\n",encoding="utf-8")
    print(f"STAGE3 COMPLETED: {total_rows} predictions, {len(full)} variant summaries; "
          f"output={out}",flush=True)


def main():
    ap=argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--out",type=Path,default=Path("mlp_stage3_results"))
    ap.add_argument("--batch-size",type=int,default=1024)
    args=ap.parse_args()
    run(args)


if __name__=="__main__":
    main()


## Pixcel_Attribution

원본: [05_model_optimize/MLP/Normal/scripts/Pixcel_Attribution.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/Pixcel_Attribution.py)  
Git blob: `bb3056dd92abc2d40efa4e34c7554d344675f6f6`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)

MLP Normal baseline XAI, stages 1-2 ONLY. Never touches official X_test/y_test.

Usage from repository root:
  python 03_experiments/model_optimization/mlp_xai_stage12.py --out xai_stage12_out

Stage 1: repeat published split, refit unchanged MLP baseline; select real
correct/misclassified images from official Validation and current Handmade.
Stage 2: calculate Integrated Gradients of the *logit*, or the logit margin
(predicted minus true) for misclassifications, with black-image baseline.
No im

CLI 인자 (정적 추출): `--out`, `--max-per-pair`, `--max-correct-per-class`

In [ ]:
%%archive_source 05_model_optimize/MLP/Normal/scripts/Pixcel_Attribution.py
#!/usr/bin/env python3
"""MLP Normal baseline XAI, stages 1-2 ONLY. Never touches official X_test/y_test.

Usage from repository root:
  python 03_experiments/model_optimization/mlp_xai_stage12.py --out xai_stage12_out

Stage 1: repeat published split, refit unchanged MLP baseline; select real
correct/misclassified images from official Validation and current Handmade.
Stage 2: calculate Integrated Gradients of the *logit*, or the logit margin
(predicted minus true) for misclassifications, with black-image baseline.
No image transformations, augmentation, hyperparameter tuning, or Test access.
"""
from __future__ import annotations

import argparse
import csv
import hashlib
import json
import os
import platform
import sys
import warnings
from collections import Counter
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import sklearn
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedGroupKFold

HERE = Path(__file__).resolve().parent
ROOT = HERE.parents[3]  # repository root; this script lives under 05_model_optimize/MLP/Normal/scripts
sys.path.insert(0, str(ROOT / "03_experiments" / "model_optimization"))
from common import fixed_split, pipeline, LABELS, DEFAULT_DATA
from mlp import build_baseline

WRITERS = (41, 42, 43, 44)
HAND_SHA = {
    41: "0a829172a72787c770139c73de34cc8b751181829ff6fe08ec260247b9be5a2f",
    42: "f364fb453d05b4fe2a705c2b8a80683b404608c7c257d664e3ec7ac86932d4c5",
    43: "10db20997c727c1bc1f1299f8e89626fadd3ec9618468d6759c86b0d2ddc4a66",
    44: "0e266a36e7b29edfcca0514836b5c383d13939948b4e335d70f61bb40bdc084d",
}
TARGET_PAIRS = [("9", "4"), ("E", "C"), ("3", "5"), ("7", "2"),
                ("5", "F"), ("4", "F"), ("B", "8"), ("8", "B"),
                ("0", "D"), ("4", "A"), ("A", "4")]
CORRECT_CLASSES = "5F94B8"
PUBLISHED_ACC = 0.9731770833333333
PUBLISHED_HANDMADE_ACC = 0.628515625


def checksum(path: Path) -> str:
    with path.open("rb") as f:
        return hashlib.file_digest(f, "sha256").hexdigest()


def read_handmade() -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    imgs, labels, writers, positions = [], [], [], []
    for w in WRITERS:
        p = ROOT / "02_data" / "Hand_made" / f"ml26_{w}.npz"
        actual_sha = checksum(p)
        if actual_sha != HAND_SHA[w]:
            raise RuntimeError(f"Handmade writer {w} checksum mismatch {actual_sha}")
        with np.load(p, allow_pickle=False) as data:
            im, y, txt = data["img"], data["label_id"], data["label"]
        if im.shape != (640, 28, 28) or im.dtype != np.uint8:
            raise RuntimeError(f"Bad Handmade image shape/dtype for writer {w}")
        if Counter(y.tolist()) != Counter({i: 40 for i in range(16)}):
            raise RuntimeError(f"Bad class balance for writer {w}")
        if any(LABELS[int(i)] != str(c) for i, c in zip(y, txt)):
            raise RuntimeError(f"Label mismatch in writer {w}")
        if w == 42 and np.unique(im).size != 256:
            raise RuntimeError("Writer 42 isn't the updated 256-level grayscale NPZ")
        imgs.append(im.reshape(640, 784).astype(np.float32))
        labels.append(y.astype(np.int64))
        writers.append(np.full(640, w, dtype=np.int32))
        positions.append(np.arange(640, dtype=np.int32))
    return tuple(np.concatenate(v) for v in (imgs, labels, writers, positions))


def official_indices(path: Path, validation_y: np.ndarray) -> np.ndarray:
    # Reconstruct exactly the first split solely from TRAIN metadata.
    with np.load(path, allow_pickle=False) as d:
        y = d["y_train"].astype(np.int64)
        src = d["train_source_index"].astype(np.int64)
        split = d["train_source_split"].astype(np.int64)
    groups = y * 1_000_000_000 + split * 100_000_000 + src
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    tr, va = next(cv.split(np.zeros((len(y), 1), np.uint8), y, groups))
    if len(tr) != 76800 or len(va) != 19200 or not np.array_equal(y[va], validation_y):
        raise RuntimeError("Original grouped split cannot be reproduced")
    return va


def prior_official_cm() -> np.ndarray:
    p = (ROOT / "03_experiments" / "W02_2026-11-09_11-15" / "results"
         / "EXP-유-003_confusion_long.csv")
    cm = np.zeros((16, 16), dtype=np.int64)
    with p.open(newline="", encoding="utf-8-sig") as f:
        for row in csv.DictReader(f):
            if row["model"] == "MLPClassifier":
                cm[LABELS.index(row["actual_label"]), LABELS.index(row["predicted_label"])] = int(row["count"])
    if cm.sum() != 19200:
        raise RuntimeError("Historical official MLP confusion matrix missing")
    return cm


def prior_handmade() -> dict[tuple[int, int], tuple[str, str]]:
    p = (ROOT / "03_experiments" / "W02_2026-11-09_11-15" / "results"
         / "EXP-유-003_rotation_v4" / "rotation_after_predictions.csv")
    previous = {}
    with p.open(newline="", encoding="utf-8-sig") as f:
        for r in csv.DictReader(f):
            if r["model"] == "MLP":
                previous[int(r["writer"]), int(r["image_idx_in_writer"])] = (r["actual"], r["predicted"])
    if len(previous) != 2560:
        raise RuntimeError("Historical Handmade MLP predictions missing")
    return previous


def get_logits(net, x):
    x = np.asarray(x, dtype=np.float64)
    a = np.maximum(x @ net.coefs_[0] + net.intercepts_[0], 0.0)
    return a @ net.coefs_[1] + net.intercepts_[1]


def integrated_gradients(net, x01: np.ndarray, pred: int, true: int, steps: int = 128):
    """Black baseline and midpoint Riemann sum for 1-hidden-layer ReLU MLP.

    For errors, the objective is logit(pred) - logit(true).
    For correct predictions, the objective is logit(true).
    The input variable is the exact model input x/255, not raw grayscale.
    """
    if len(net.coefs_) != 2 or net.activation != "relu":
        raise RuntimeError("Analytic IG only supports the unchanged 1-layer ReLU MLP")
    x = np.asarray(x01, dtype=np.float64).reshape(784)
    w0, w1 = (np.asarray(a, dtype=np.float64) for a in net.coefs_)
    b0 = np.asarray(net.intercepts_[0], dtype=np.float64)
    out_weight = w1[:, pred].copy()
    if pred != true:
        out_weight -= w1[:, true]
    alpha = (np.arange(steps, dtype=np.float64) + 0.5) / steps
    z = (alpha[:, None] * x[None, :]) @ w0 + b0
    active = (z > 0.0).astype(np.float64)
    grad = (active * out_weight[None, :]) @ w0.T
    attribution = x * grad.mean(axis=0)
    logits_x, logits_0 = get_logits(net, x), get_logits(net, np.zeros_like(x))
    if pred == true:
        delta = logits_x[pred] - logits_0[pred]
        objective = "true_class_logit"
    else:
        delta = ((logits_x[pred] - logits_x[true])
                 - (logits_0[pred] - logits_0[true]))
        objective = "pred_minus_true_logit_margin"
    return attribution.reshape(28, 28), objective, float(delta), float(attribution.sum() - delta)


def diverse_selection(indices: np.ndarray, writers: np.ndarray, count: int, separate: bool):
    if not separate:
        return list(indices[:count])
    chosen = []
    # Favor different writers for each confusion pair.
    buckets = {w: indices[writers[indices] == w].tolist() for w in WRITERS}
    order = sorted(WRITERS, key=lambda w: (-len(buckets[w]), w))
    while len(chosen) < count and any(buckets.values()):
        for w in order:
            if len(chosen) >= count:
                break
            if buckets[w]:
                chosen.append(buckets[w].pop(0))
    return chosen


def select_images(domain: str, x: np.ndarray, y: np.ndarray, pred: np.ndarray,
                  writer: np.ndarray, index: np.ndarray, prob: np.ndarray, net,
                  max_pairs: int, max_correct: int):
    picked, used = [], set()
    for a, b in TARGET_PAIRS:
        matches = np.flatnonzero((y == LABELS.index(a)) & (pred == LABELS.index(b)))
        for k in diverse_selection(matches, writer, max_pairs, domain == "Handmade"):
            if int(k) not in used:
                picked.append((int(k), f"{a}_to_{b}"))
                used.add(int(k))
    for c in CORRECT_CLASSES:
        matches = np.flatnonzero((y == LABELS.index(c)) & (pred == LABELS.index(c)))
        for k in diverse_selection(matches, writer, max_correct, domain == "Handmade"):
            if int(k) not in used:
                picked.append((int(k), f"correct_{c}"))
                used.add(int(k))
    rows, views, raw, attribution = [], [], [], []
    for k, kind in picked:
        actual, prediction = int(y[k]), int(pred[k])
        x01 = x[k].astype(np.float64) / 255.0
        attr, objective, delta, residual = integrated_gradients(net, x01, prediction, actual)
        case = {
            "domain": domain, "case": kind, "writer": int(writer[k]),
            "index": int(index[k]), "true": LABELS[actual], "predicted": LABELS[prediction],
            "correct": int(actual == prediction),
            "p_true": float(prob[k, actual]), "p_predicted": float(prob[k, prediction]),
            "objective": objective, "IG_sum": float(attr.sum()),
            "logit_objective_delta": delta, "IG_completeness_residual": residual,
            "IG_abs_sum": float(np.abs(attr).sum()),
        }
        rows.append(case)
        views.append({"row": case, "img": x[k].reshape(28,28).copy(), "ig": attr.copy()})
        raw.append(x[k].reshape(28, 28).astype(np.uint8))
        attribution.append(attr.astype(np.float32))
    return rows, views, raw, attribution


def save_plots(views: list[dict], out: Path, batch: int = 6):
    plot_dir = out / "review_grids"
    plot_dir.mkdir(exist_ok=True)
    for start in range(0, len(views), batch):
        page = views[start:start+batch]
        fig, axes = plt.subplots(len(page), 3, figsize=(10.7, 2.7 * len(page)), squeeze=False)
        for j, case in enumerate(page):
            r, img, att = case["row"], case["img"], case["ig"]
            maxabs = max(float(np.max(np.abs(att))), 1e-8)
            axes[j,0].imshow(img, cmap="gray", vmin=0, vmax=255)
            axes[j,1].imshow(att, cmap="RdBu_r", vmin=-maxabs, vmax=maxabs)
            axes[j,2].imshow(np.abs(att), cmap="magma", vmin=0, vmax=maxabs)
            writer_desc = f"writer {r['writer']} #" if r["writer"] >= 0 else "train index #"
            axes[j,0].set_title(f"{r['domain']}: {r['true']} -> {r['predicted']}  ({writer_desc}{r['index']})", fontsize=9)
            axes[j,1].set_title("Signed IG (red = supports objective)", fontsize=9)
            axes[j,2].set_title("Absolute IG", fontsize=9)
            for ax in axes[j]:
                ax.set_xticks([])
                ax.set_yticks([])
        fig.tight_layout()
        fig.savefig(plot_dir / f"cases_{start//batch+1:02d}.png", dpi=135)
        plt.close(fig)


def save_focus_grid(views: list[dict], wanted: list[tuple], output: Path):
    """Create reproducible manuscript-friendly views from verified selected cases."""
    picked = []
    for domain, true, predicted, writer, sample_idx in wanted:
        matching = [v for v in views if
            v["row"]["domain"] == domain and
            v["row"]["true"] == true and
            v["row"]["predicted"] == predicted and
            (writer is None or v["row"]["writer"] == writer) and
            (sample_idx is None or v["row"]["index"] == sample_idx)]
        if not matching:
            raise RuntimeError(f"Missing focus-grid input: {(domain, true, predicted, writer, sample_idx)}")
        picked.append(matching[0])
    fig, axes = plt.subplots(len(picked), 3, figsize=(11.4, 2.35 * len(picked)),
                             squeeze=False)
    for j, case in enumerate(picked):
        meta, img, attr = case["row"], case["img"], case["ig"]
        bound = max(float(np.abs(attr).max()), 1e-8)
        axes[j, 0].imshow(img, cmap="gray", vmin=0, vmax=255)
        axes[j, 1].imshow(attr, cmap="RdBu_r", vmin=-bound, vmax=bound)
        axes[j, 2].imshow(np.abs(attr), cmap="magma", vmin=0, vmax=bound)
        case_desc = (f"writer {meta['writer']} / image {meta['index']}" if meta["writer"] >= 0
                     else f"official train index {meta['index']}")
        axes[j, 0].set_title(f"{meta['domain']}  {meta['true']} -> {meta['predicted']}\n{case_desc}", fontsize=9)
        axes[j, 1].set_title("Signed Integrated Gradients", fontsize=9)
        axes[j, 2].set_title("Absolute Integrated Gradients", fontsize=9)
        for ax in axes[j]:
            ax.set_xticks([])
            ax.set_yticks([])
    fig.tight_layout()
    fig.savefig(output, dpi=150)
    plt.close(fig)


def write_csv(path: Path, rows: list[dict]):
    if not rows:
        return
    with path.open("w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--out", type=Path, default=Path("mlp_xai_stage12_results"))
    parser.add_argument("--max-per-pair", type=int, default=4)
    parser.add_argument("--max-correct-per-class", type=int, default=2)
    args = parser.parse_args()
    out = args.out.resolve()
    out.mkdir(parents=True, exist_ok=True)
    print("Loading official TRAIN split (never reading X_test/y_test)", flush=True)
    xtr, ytr, xva, yva, official_sha = fixed_split(DEFAULT_DATA)
    vaidx = official_indices(DEFAULT_DATA, yva)
    print("Loading current Handmade image arrays", flush=True)
    xhm, yhm, whm, ihm = read_handmade()
    estimator, is_scaled = build_baseline()
    if not is_scaled or estimator.get_params()["random_state"] != 42:
        raise RuntimeError("Unexpected changed baseline definition")
    model = pipeline(estimator, is_scaled)
    with warnings.catch_warnings(record=True) as ws:
        warnings.simplefilter("always")
        print("Fitting unchanged original MLP Baseline; no Handmade in fit()", flush=True)
        model.fit(xtr, ytr)
    pv = model.predict(xva).astype(np.int64)
    ph = model.predict(xhm).astype(np.int64)
    historical = prior_handmade()
    hand_disagreements = 0
    for k, (w, idx) in enumerate(zip(whm, ihm)):
        a, b = historical[int(w), int(idx)]
        if a != LABELS[int(yhm[k])]:
            raise RuntimeError(f"Handmade label mismatch at {w}, {idx}")
        hand_disagreements += b != LABELS[int(ph[k])]
    cm = confusion_matrix(yva, pv, labels=np.arange(16))
    cm_historical_l1 = int(np.abs(cm - prior_official_cm()).sum())
    av, ah = float(accuracy_score(yva, pv)), float(accuracy_score(yhm, ph))
    print(f"Official accuracy={av:.8f}, historical={PUBLISHED_ACC:.8f}, CM_L1={cm_historical_l1}", flush=True)
    print(f"Handmade accuracy={ah:.8f}, historical={PUBLISHED_HANDMADE_ACC:.8f}, predictions differing={hand_disagreements}", flush=True)
    net = model.named_steps["model"]
    if net.classes_.tolist() != list(range(16)):
        raise RuntimeError("Unexpected MLP class order")

    pvproba, phproba = model.predict_proba(xva), model.predict_proba(xhm)
    official_writer = np.full(len(yva), -1, dtype=np.int32)
    print("Selecting correct/errors and calculating integrated gradients", flush=True)
    o_rows, o_views, o_raw, o_attrs = select_images(
        "OfficialVal", xva, yva, pv, official_writer, vaidx, pvproba, net,
        args.max_per_pair, args.max_correct_per_class)
    h_rows, h_views, h_raw, h_attrs = select_images(
        "Handmade", xhm, yhm, ph, whm, ihm, phproba, net,
        args.max_per_pair, args.max_correct_per_class)
    cases = o_rows + h_rows
    write_csv(out / "selected_cases.csv", cases)
    write_csv(out / "handmade_all_predictions.csv", [
        {"writer": int(w), "index": int(idx), "true": LABELS[int(a)],
         "predicted": LABELS[int(b)], "correct": int(a==b)}
        for w,idx,a,b in zip(whm, ihm, yhm, ph)])
    np.savez_compressed(out / "selected_pixels_and_IG.npz",
        pixels=np.stack(o_raw+h_raw), IG=np.stack(o_attrs+h_attrs))
    save_plots(o_views+h_views, out)
    focus = o_views + h_views
    save_focus_grid(focus, [
        ("Handmade", "5", "F", 43, 53),
        ("Handmade", "B", "8", 42, 27),
        ("Handmade", "9", "4", 41, 25),
    ], out / "MLP_Pixcel_Attribution_handmade_examples.png")
    save_focus_grid(focus, [
        ("OfficialVal", "9", "4", None, None),
        ("OfficialVal", "E", "C", None, None),
        ("OfficialVal", "3", "5", None, None),
        ("Handmade", "5", "F", 43, 53),
        ("Handmade", "B", "8", 42, 27),
        ("Handmade", "9", "4", 41, 25),
    ], out / "MLP_Pixcel_Attribution_6_examples.png")

    if cm_historical_l1 != 0 or hand_disagreements != 0:
        raise RuntimeError("Retrained baseline predictions do not match the published W02 results")

    metadata = {
        "scope": "Stage1 extract + Stage2 Integrated Gradients ONLY; not Stage3",
        "test_data_used": False,
        "handmade_used_for_training": False,
        "data_sha256": official_sha, "handmade_sha256": HAND_SHA,
        "official_split": "StratifiedGroupKFold(5, shuffle=True, random_state=42), first fold",
        "training_n": int(len(ytr)), "validation_n": int(len(yva)),
        "handmade_n": int(len(yhm)), "official_accuracy": av,
        "historical_official_accuracy": PUBLISHED_ACC,
        "official_confusion_matrix_cell_L1_difference_from_historical": cm_historical_l1,
        "official_accuracy_exact_match": bool(av == PUBLISHED_ACC),
        "handmade_accuracy": ah, "historical_handmade_accuracy": PUBLISHED_HANDMADE_ACC,
        "handmade_prediction_mismatch_count": int(hand_disagreements),
        "historical_predictions_exact_match": bool(cm_historical_l1 == 0 and hand_disagreements == 0),
        "official_macro_f1": float(f1_score(yva, pv, average="macro")),
        "handmade_macro_f1": float(f1_score(yhm, ph, average="macro")),
        "selected_n": len(cases), "selected_official_n": len(o_rows),
        "selected_handmade_n": len(h_rows),
        "XAI_method": "128-step midpoint Integrated Gradients; black baseline; class logit or predicted-vs-true logit margin",
        "IG_completeness_residual_max_abs": float(max(abs(r["IG_completeness_residual"]) for r in cases)),
        "model_params": {k: repr(v) for k,v in net.get_params().items()},
        "warnings": [f"{w.category.__name__}: {w.message}" for w in ws],
        "versions": {"python":platform.python_version(),"numpy":np.__version__,"sklearn":sklearn.__version__,
                     "matplotlib":matplotlib.__version__},
    }
    (out / "run_metadata.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
    description = [
        "# Normal MLP baseline: Stage 1+2 XAI report",
        "",
        "## Reproduction check",
        f"- Official: {len(ytr):,} train / {len(yva):,} Validation; accuracy {av:.8%} (published {PUBLISHED_ACC:.8%}).",
        f"- Official CM cell L1 difference from historical: **{cm_historical_l1}**.",
        f"- Handmade: {len(yhm):,} development-only images; accuracy {ah:.8%} (published {PUBLISHED_HANDMADE_ACC:.8%}).",
        f"- Handmade prediction disagreements versus published CSV: **{hand_disagreements} / {len(yhm)}**.",
        f"- Selected images with actual pixels and IG: {len(cases)}, Official {len(o_rows)}, Handmade {len(h_rows)}.",
        f"- Max Integrated Gradients completeness residual: {metadata['IG_completeness_residual_max_abs']:.4g}.",
        "",
        "## Read the outputs",
        "- `selected_cases.csv`: exact image provenance and objective/probabilities.",
        "- `review_grids/*.png`: original 28x28 images + signed and absolute pixel attributions.",
        "- `selected_pixels_and_IG.npz`: matching images and attribution maps, same order as selected_cases.csv.",
        "- `handmade_all_predictions.csv`: fresh predictions for 2,560 images.",
        "- `run_metadata.json`: checksums, accuracy, reproduction deviations, software versions.",
        "",
        "## Interpretation limits",
        "Attribution shows contributions to logits (or predicted-versus-true margin), **not proof** that",
        "the model relies on shape versus brightness/position. Causal confirmation via controlled",
        "image transforms is deliberately deferred to Stage 3 after the user's approval.",
        "Official X_test/y_test were never read. No hand-written images were used in fit().",
        "If historical predictions differ, interpret all attributions as **the retrained baseline**,",
        "not as the historically saved model's exact weights.",
    ]
    (out / "SUMMARY.md").write_text("\n".join(description)+"\n", encoding="utf-8")
    print(f"DONE: {out} ({len(cases)} selected), only stages 1 and 2", flush=True)


if __name__ == "__main__":
    main()


## plot_input_sensitivity

원본: [05_model_optimize/MLP/Normal/scripts/plot_input_sensitivity.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/plot_input_sensitivity.py)  
Git blob: `4cd6c5935e890900ca426012fbc6ec88ee43c734`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)

Recreate Stage 3 descriptive plots from the actual result CSVs (no retraining).

CLI 인자 (정적 추출): `--result-dir`

In [ ]:
%%archive_source 05_model_optimize/MLP/Normal/scripts/plot_input_sensitivity.py
#!/usr/bin/env python3
"""Recreate Stage 3 descriptive plots from the actual result CSVs (no retraining)."""
from __future__ import annotations

import argparse
import csv
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

LABELS = "0123456789ABCDEF"


def rows(file: Path):
    with file.open("r", encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--result-dir", required=True, type=Path)
    args = parser.parse_args()
    d = args.result_dir.resolve()
    overview = rows(d / "stage3_sensitivity_summary.csv")
    perclass = rows(d / "stage3_per_class.csv")
    writer = rows(d / "stage3_per_writer.csv")
    variants = [x["variant"] for x in overview if x["domain"] == "Handmade" and x["variant"] != "original"]
    assert len(variants) == 20 and len(set(variants)) == 20

    # Comparison over official and Handmade development data.
    data = {(r["domain"], r["variant"]): r for r in overview}
    x = np.arange(len(variants))
    fig, ax = plt.subplots(figsize=(15, 6.5))
    ax.bar(x - 0.21, [float(data["OfficialVal", v]["accuracy_delta_pp"]) for v in variants],
           width=0.42, label="Official Validation (19,200)")
    ax.bar(x + 0.21, [float(data["Handmade", v]["accuracy_delta_pp"]) for v in variants],
           width=0.42, label="Hand-made (2,560)")
    ax.axhline(0, linewidth=0.8, color="gray")
    ax.set_xticks(x, variants, rotation=65, ha="right", fontsize=8)
    ax.set_ylabel("Accuracy change against unchanged input (percentage points)")
    ax.set_title("Normal MLP: One-variable Input Sensitivity (20 perturbations)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(d / "STAGE3_Accuracy_Change_Comparison.png", dpi=155)
    plt.close(fig)

    # Class-balanced digits 0-9, letters A-F on Handmade only.
    bykey = {(r["domain"], r["variant"], r["group"]): r for r in perclass}
    digit, letter = [], []
    for v in variants:
        vals = []
        for classes in (LABELS[:10], LABELS[10:]):
            current = [bykey["Handmade", v, c] for c in classes]
            original = [bykey["Handmade", "original", c] for c in classes]
            n = sum(int(row["n"]) for row in current)
            ncorrect = sum(round(float(row["accuracy"]) * int(row["n"])) for row in current)
            ncorrect_orig = sum(round(float(row["accuracy"]) * int(row["n"])) for row in original)
            vals.append(100 * (ncorrect - ncorrect_orig) / n)
        digit.append(vals[0]); letter.append(vals[1])
    fig, ax = plt.subplots(figsize=(15, 6.5))
    ax.bar(x - 0.21, digit, width=0.42, label="Digits (0–9)")
    ax.bar(x + 0.21, letter, width=0.42, label="Letters (A–F)")
    ax.axhline(0, linewidth=0.8, color="gray")
    ax.set_xticks(x, variants, rotation=65, ha="right", fontsize=8)
    ax.set_ylabel("Hand-made accuracy change (percentage points)")
    ax.set_title("Normal MLP: Opposing Effects Across Digits and Letters")
    ax.legend()
    fig.tight_layout()
    fig.savefig(d / "STAGE3_Digit_Letter_Tradeoff.png", dpi=155)
    plt.close(fig)

    # Writer differences on variants with clear overall effects.
    selection = ["shift_L1", "shift_R1", "shift_D1", "scale_090", "scale_095",
                 "stroke_thin_25", "stroke_thick_40"]
    bywriter = {(r["variant"], r["group"]): r for r in writer}
    xx = np.arange(len(selection))
    fig, ax = plt.subplots(figsize=(12, 5.8))
    for w in ("41", "42", "43", "44"):
        values = [float(bywriter[(v, w)]["accuracy_delta_pp"]) for v in selection]
        ax.plot(xx, values, marker="o", label=f"Writer {w}")
    ax.set_xticks(xx, selection, rotation=30, ha="right")
    ax.axhline(0, linewidth=0.8, color="gray")
    ax.set_ylabel("Accuracy change (percentage points)")
    ax.set_title("Normal MLP: Hand-made Input Sensitivity by Writer")
    ax.legend()
    fig.tight_layout()
    fig.savefig(d / "STAGE3_Per_Writer.png", dpi=155)
    plt.close(fig)

    print("Created 3 descriptive plots from CSV only", flush=True)


if __name__ == "__main__":
    main()


## Spatial_Response_Analysis

원본: [05_model_optimize/MLP/Normal/scripts/Spatial_Response_Analysis.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/scripts/Spatial_Response_Analysis.py)  
Git blob: `f839894c1e1dae39b4f9cff4f8857d7707d3b61d`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Normal/logs)

EXP유003 Spatial Distribution & MLP Response Analysis (diagnostic, no tuning).

Uses the EXACT frozen Normal MLP from completed EXP유002 Stage 3.
Studies Official Train/Validation + current Handmade geometry and relates
paired transformations to prediction flips, hidden response and fixed-logit
margin Integrated Gradients. Official Test is NEVER loaded.

Outputs are GitHub Actions artifacts for review; this script does not commit
or publish results. Features/ground-truth labels are used for invest

CLI 인자 (정적 추출): `--out`, `--batch`

In [ ]:
%%archive_source 05_model_optimize/MLP/Normal/scripts/Spatial_Response_Analysis.py
#!/usr/bin/env python3
"""EXP유003 Spatial Distribution & MLP Response Analysis (diagnostic, no tuning).

Uses the EXACT frozen Normal MLP from completed EXP유002 Stage 3.
Studies Official Train/Validation + current Handmade geometry and relates
paired transformations to prediction flips, hidden response and fixed-logit
margin Integrated Gradients. Official Test is NEVER loaded.

Outputs are GitHub Actions artifacts for review; this script does not commit
or publish results. Features/ground-truth labels are used for investigation
only, NOT for any inference-time preprocessing rule.
"""
from __future__ import annotations

import argparse
from collections import defaultdict
import csv
import hashlib
import gzip
import json
import math
import platform
import sys
import time
from pathlib import Path
import warnings

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import scipy
from scipy.stats import wasserstein_distance
import sklearn
from sklearn.metrics import accuracy_score, confusion_matrix

HERE = Path(__file__).resolve().parent
ROOT = HERE.parents[3]
sys.path.insert(0, str(HERE))
from Pixcel_Attribution import (fixed_split, DEFAULT_DATA, official_indices,
    prior_official_cm, prior_handmade, read_handmade, LABELS)
from Input_Sensitivity import transform, VARIANTS, get_output, margin, select_focus

STAGE3_DIR = ROOT / "05_model_optimize" / "MLP" / "Normal" / "results" / "Input_Sensitivity"
MODEL_PATH = STAGE3_DIR / "normal_mlp_refit_stage3.joblib"
PRIOR_METADATA = STAGE3_DIR / "stage3_run_metadata.json"
COMPARISON = ("original", "shift_L1", "shift_R1", "shift_D1", "scale_090", "scale_095")
SCATTER_CLASSES = ("5", "9", "B", "F")
MAIN_THRESHOLD = 32
THRESHOLDS = (32, 64, 128)


def csvwrite(file, rows):
    if not rows:
        return
    with file.open("w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)


def metrics(images: np.ndarray, threshold: int = 32):
    """Fixed absolute grayscale threshold; vectorized (n,28,28)."""
    a = images.reshape(-1, 28, 28).astype(np.float64, copy=False)
    m = a >= threshold
    active = m.sum(axis=(1, 2)).astype(np.int32)
    valid = active > 0
    xn = m.any(axis=1)
    yn = m.any(axis=2)
    xmin = np.where(valid, np.argmax(xn, axis=1), -1)
    xmax = np.where(valid, 27 - np.argmax(xn[:, ::-1], axis=1), -1)
    ymin = np.where(valid, np.argmax(yn, axis=1), -1)
    ymax = np.where(valid, 27 - np.argmax(yn[:, ::-1], axis=1), -1)
    w = np.where(valid, xmax - xmin + 1, np.nan).astype(float)
    h = np.where(valid, ymax - ymin + 1, np.nan).astype(float)
    wt = a * m
    mass = wt.sum(axis=(1, 2))
    mass_safe = np.maximum(mass, 1e-30)
    cx = (wt * np.arange(28)[None, None, :]).sum(axis=(1, 2)) / mass_safe
    cy = (wt * np.arange(28)[None, :, None]).sum(axis=(1, 2)) / mass_safe
    cx = np.where(valid, cx, np.nan)
    cy = np.where(valid, cy, np.nan)
    return {
      "cx":cx, "cy":cy,
      "bbox_cx":np.where(valid,(xmin+xmax)/2,np.nan),
      "bbox_cy":np.where(valid,(ymin+ymax)/2,np.nan),
      "width":w, "height":h,
      "left_margin":np.where(valid,xmin,np.nan).astype(float),
      "right_margin":np.where(valid,27-xmax,np.nan).astype(float),
      "top_margin":np.where(valid,ymin,np.nan).astype(float),
      "bottom_margin":np.where(valid,27-ymax,np.nan).astype(float),
      "aspect":w/np.maximum(h,1),
      "ink_frac":active/784,
      "mass":mass,
      "valid":valid,
    }


def shape_features(x: np.ndarray, thresholds=THRESHOLDS):
    parts = {t: metrics(x, t) for t in thresholds}
    n = len(x)
    result = {}
    for k,v in parts[MAIN_THRESHOLD].items():
        result[k] = v
    for t in thresholds:
        if t == MAIN_THRESHOLD:
            continue
        for key in ("cx","cy","bbox_cx","bbox_cy","width","height","ink_frac"):
            result[f"{key}_t{t}"] = parts[t][key]
        result[f"valid_t{t}"] = parts[t]["valid"]
    assert all(len(v)==n for v in result.values())
    return result


def geometry_matrix(feats):
    return np.column_stack([feats[x] for x in ("cx","cy","width","height")])


def reference_stats(xtrain, ytrain):
    center=np.zeros((16,4),dtype=np.float64)
    scale=np.zeros((16,4),dtype=np.float64)
    rows=[]
    a=geometry_matrix(xtrain)
    for i,c in enumerate(LABELS):
        v=a[(ytrain==i) & np.isfinite(a).all(axis=1)]
        if len(v)<100:raise RuntimeError(f"Insufficient official train reference for class {c}")
        med=np.median(v,axis=0)
        q1,q3=np.percentile(v,[25,75],axis=0)
        robust=np.maximum((q3-q1)/1.349,0.75)
        center[i],scale[i]=med,robust
        rows.append({
            "class":c,"n":len(v),
            **{f"median_{k}":float(med[j]) for j,k in enumerate(("cx","cy","width","height"))},
            **{f"scale_{k}":float(robust[j]) for j,k in enumerate(("cx","cy","width","height"))},
        })
    return center,scale,rows


def reference_distance(f,y,center,scale):
    v=geometry_matrix(f)
    with np.errstate(invalid="ignore"):
        d=np.sqrt(np.square((v-center[y])/scale[y]).sum(axis=1))
    return d


def describe(f,y,domain,writer=-1,aug=None):
    data=[]
    for cls in ["ALL"]+list(LABELS):
        mask=np.ones(len(y),dtype=bool) if cls=="ALL" else (y==LABELS.index(cls))
        valid=mask & f["valid"]
        row={"dataset":domain,"writer":writer,"class":cls,
             "n":int(mask.sum()),"valid_n":int(valid.sum())}
        for key in ("cx","cy","bbox_cx","bbox_cy","width","height","ink_frac",
                    "left_margin","right_margin","top_margin","bottom_margin"):
            v=f[key][valid]
            row[f"{key}_mean"]=float(np.mean(v)) if len(v) else None
            row[f"{key}_median"]=float(np.median(v)) if len(v) else None
            row[f"{key}_p10"]=float(np.percentile(v,10)) if len(v) else None
            row[f"{key}_p90"]=float(np.percentile(v,90)) if len(v) else None
        data.append(row)
    return data


def write_features(file, domain, x, y, writer, index, augmented=None, batch=1024):
    """One compact row per source image, with threshold-robustness columns."""
    fields=["dataset","writer","sample_index","class","augmented"]
    keys=list(shape_features(x[:1]).keys())
    write_header = not file.exists() or file.stat().st_size == 0
    with gzip.open(file,"at",newline="",encoding="utf-8") as stream:
        csvwriter=csv.DictWriter(stream,fieldnames=fields+keys)
        if write_header:
            csvwriter.writeheader()
        for beg in range(0,len(y),batch):
            end=min(beg+batch,len(y))
            f=shape_features(x[beg:end])
            for j in range(end-beg):
                row={"dataset":domain, "writer":int(writer[beg+j]),
                     "sample_index":int(index[beg+j]),
                     "class":LABELS[int(y[beg+j])],
                     "augmented":int(augmented[beg+j]) if augmented is not None else ""}
                for k in keys:
                    val=f[k][j]
                    row[k]=(float(val) if np.isfinite(val) else "")
                csvwriter.writerow(row)


def read_stage3_summary():
    file=STAGE3_DIR/"stage3_sensitivity_summary.csv"
    with file.open(encoding="utf-8-sig",newline="") as f:
        return {(r["domain"],r["variant"]):r for r in csv.DictReader(f)
                if r["group_type"]=="all"}


def top_hidden(model,raw):
    net=model.named_steps["model"]
    a=np.asarray(raw,dtype=np.float32)/np.float32(255)
    return np.maximum(a@net.coefs_[0]+net.intercepts_[0],0.0)


def hidden_change(old,new):
    norm0=np.linalg.norm(old,axis=1)
    norm1=np.linalg.norm(new,axis=1)
    similarity=(old*new).sum(axis=1)/np.maximum(norm0*norm1,1e-8)
    relative_l2=np.linalg.norm(new-old,axis=1)/np.maximum(norm0,1e-8)
    switched=np.mean((old>0)!=(new>0),axis=1)
    return similarity,relative_l2,switched


def ig_true_margin(net, raw, true_cls, comparison_cls, steps=128):
    """Signed IG of the SAME fixed true-minus-competitor logit objective."""
    assert true_cls != comparison_cls
    x=np.asarray(raw,dtype=np.float64).reshape(784)/255.0
    w0=np.asarray(net.coefs_[0],dtype=np.float64)
    b0=np.asarray(net.intercepts_[0],dtype=np.float64)
    w1=np.asarray(net.coefs_[1],dtype=np.float64)
    bias1=np.asarray(net.intercepts_[1],dtype=np.float64)
    diff=w1[:,true_cls]-w1[:,comparison_cls]
    alpha=(np.arange(steps,dtype=np.float64)+0.5)/steps
    z=(alpha[:,None]*x[None,:])@w0+b0
    grad=(((z>0)*diff[None,:])@w0.T).mean(axis=0)
    ig=x*grad
    def objective(v):
        hidden=np.maximum(v@w0+b0,0)
        return hidden@diff+bias1[true_cls]-bias1[comparison_cls]
    delta=float(objective(x)-objective(np.zeros_like(x)))
    residual=float(ig.sum()-delta)
    return ig.reshape(28,28),delta,residual


def focus_analysis(focus,model, variants,out):
    net=model.named_steps["model"]
    records=[]
    prepared=[]
    vmap={v.name:v for v in variants}
    for i, ex in enumerate(focus):
        original=ex["pixels"].reshape(1,784)
        z0,p0=get_output(model,original)
        h0=top_hidden(model,original)
        assert int(z0.argmax(axis=1)[0])==ex["original_pred"]
        y=int(ex["true"])
        alternate = int(ex["original_pred"])
        if alternate==y:
            competitors=z0[0].copy()
            competitors[y]=-np.inf
            alternate=int(competitors.argmax())
        for v in variants:
            altered=transform(original,v)
            logits,prob=get_output(model,altered)
            h=top_hidden(model,altered)
            cs,rel,sw=hidden_change(h0,h)
            pred=int(logits[0].argmax())
            geom=metrics(altered)
            prepared.append((i,ex,v,altered,prob,logits))
            records.append({
                "focus_id":i,"domain":ex["domain"],"writer":int(ex["writer"]),
                "index":int(ex["index"]), "true":LABELS[y],
                "original_pred":LABELS[ex["original_pred"]],
                "original_correct":int(y==ex["original_pred"]),
                "new_pred":LABELS[pred],"new_correct":int(y==pred),
                "variant":v.name,
                "p_true":float(prob[0,y]),
                "p_fixed_alternate":float(prob[0,alternate]),
                "fixed_true_minus_alternate_logit":float(logits[0,y]-logits[0,alternate]),
                "hidden_cosine_to_original":float(cs[0]),
                "hidden_relative_l2":float(rel[0]),
                "hidden_changed_active_fraction":float(sw[0]),
                "cx":float(geom["cx"][0]),"cy":float(geom["cy"][0]),
                "width":float(geom["width"][0]),"height":float(geom["height"][0]),
            })
    csvwrite(out/"focus_hidden_responses.csv",records)
    # Balanced qualitative audit: include original errors, corrected errors,
    # stable errors and formerly correct examples. Do not cherry-pick only successes.
    chosen=[]
    for direction in (("5","F"),("B","8"),("9","F"),("9","4"),
                      ("0","D"),("7","F")):
        matches=[(i,x) for i,x in enumerate(focus)
                 if x["domain"]=="Handmade" and LABELS[x["true"]]==direction[0]
                 and LABELS[x["original_pred"]]==direction[1]]
        if matches:
            chosen.append(matches[0][0])
            if len(matches)>1:chosen.append(matches[1][0])
    for cls in ("5","9","B","F","8","4"):
        matches=[i for i,x in enumerate(focus)
                 if x["domain"]=="Handmade" and LABELS[x["true"]]==cls
                 and x["true"]==x["original_pred"]]
        if matches:chosen.append(matches[0])
    chosen=list(dict.fromkeys(chosen))[:18]
    ig_variants=("original","shift_R1","shift_D1","scale_095")
    ig_records=[]
    image_arrays=[]
    ig_arrays=[]
    for i in chosen:
        ex=focus[i]
        y=int(ex["true"])
        alternate=int(ex["original_pred"])
        if alternate==y:
            logits,_=get_output(model,ex["pixels"].reshape(1,784))
            rivals=logits[0].copy()
            rivals[y]=-np.inf
            alternate=int(rivals.argmax())
        for name in ig_variants:
            raw=transform(ex["pixels"].reshape(1,784),vmap[name])
            ig,delta,residual=ig_true_margin(net,raw,y,alternate)
            ig_records.append({
                "focus_id":i, "domain":ex["domain"],"writer":ex["writer"],
                "index":ex["index"], "true":LABELS[y],"fixed_competitor":LABELS[alternate],
                "variant":name,"objective":"true_minus_fixed_competitor_logit",
                "ig_sum":float(ig.sum()),"objective_delta_from_black":delta,
                "completeness_residual":residual,"ig_absolute_sum":float(np.abs(ig).sum()),
            })
            image_arrays.append(raw.reshape(28,28).astype(np.uint8))
            ig_arrays.append(ig.astype(np.float32))
    csvwrite(out/"focus_ig_completeness.csv",ig_records)
    np.savez_compressed(out/"focus_ig_maps.npz",
                        pixels=np.stack(image_arrays),signed_IG=np.stack(ig_arrays))
    figdir=out/"focus_ig_grids"
    figdir.mkdir(exist_ok=True)
    for page in range(0,len(chosen),4):
        picks=chosen[page:page+4]
        fig,axs=plt.subplots(len(picks),len(ig_variants)*2,
                             figsize=(18,3.2*len(picks)),squeeze=False)
        for ridx,i in enumerate(picks):
            ex=focus[i]
            for j,name in enumerate(ig_variants):
                k=(chosen.index(i))*len(ig_variants)+j
                raw=image_arrays[k]
                ig=ig_arrays[k]
                bound=max(float(np.max(np.abs(ig))),1e-8)
                a=axs[ridx,2*j]
                b=axs[ridx,2*j+1]
                a.imshow(raw,cmap="gray",vmin=0,vmax=255)
                b.imshow(ig,cmap="RdBu_r",vmin=-bound,vmax=bound)
                a.set_title(f"{LABELS[ex['true']]} / orig->{LABELS[ex['original_pred']]}\n{name} image",fontsize=8)
                b.set_title("IG: true - fixed rival",fontsize=8)
                a.set_xticks([]);a.set_yticks([])
                b.set_xticks([]);b.set_yticks([])
        fig.tight_layout()
        fig.savefig(figdir/f"focus_{page//4+1:02d}.png",dpi=115)
        plt.close(fig)
    return records,ig_records


def run(args):
    started=time.monotonic()
    assert MAIN_THRESHOLD in THRESHOLDS
    out=args.out.resolve()
    out.mkdir(parents=True,exist_ok=True)
    with PRIOR_METADATA.open(encoding="utf-8") as f:
        oldmeta=json.load(f)
    assert oldmeta["total_prediction_rows"]==456960
    assert oldmeta["baseline_validation_cm_L1_difference"]==0
    assert oldmeta["baseline_handmade_prediction_difference"]==0
    model=joblib.load(MODEL_PATH)  # Our own verified GitHub Actions saved checkpoint.
    print("Loaded VERIFIED frozen Stage 3 Normal MLP checkpoint",flush=True)

    xtr,ytr,xva,yva,off_sha=fixed_split(DEFAULT_DATA)
    vaidx=official_indices(DEFAULT_DATA,yva)
    with np.load(DEFAULT_DATA,allow_pickle=False) as z:
        aug=z["train_is_augmented"]
    official_index=np.arange(96000)
    train_idx=np.setdiff1d(official_index,vaidx,assume_unique=True)
    assert len(train_idx)==76800
    training_aug=aug[train_idx]
    xhm,yhm,whm,ih=read_handmade()
    pva=model.predict(xva)
    phm=model.predict(xhm)
    official_l1=int(np.abs(confusion_matrix(yva,pva,labels=np.arange(16))
                       -prior_official_cm()).sum())
    previous=prior_handmade()
    handmade_difference=sum(previous[(int(w),int(idx))] !=
                            (LABELS[int(a)],LABELS[int(b)])
                            for w,idx,a,b in zip(whm,ih,yhm,phm))
    assert official_l1==0 and handmade_difference==0
    assert abs(accuracy_score(yva,pva)-oldmeta["original_accuracy"]["OfficialVal"])<1e-12
    assert abs(accuracy_score(yhm,phm)-oldmeta["original_accuracy"]["Handmade"])<1e-12
    print("Reused frozen model: both prior confusion/prediction validations exactly match",flush=True)

    # All geometry measurements are computed on RAW source input, not transformed.
    print("Extracting Official Train/Val and 4-writer geometry",flush=True)
    ftr=shape_features(xtr)
    fva=shape_features(xva)
    fhm=shape_features(xhm)
    ftr_original={k:v[~training_aug.astype(bool)] for k,v in ftr.items()}
    yt_original=ytr[~training_aug.astype(bool)]
    ftr_aug={k:v[training_aug.astype(bool)] for k,v in ftr.items()}
    yt_aug=ytr[training_aug.astype(bool)]
    statrows=[]
    statrows+=describe(ftr,ytr,"OfficialTrain")
    statrows+=describe(ftr_original,yt_original,"OfficialTrain_Original")
    statrows+=describe(ftr_aug,yt_aug,"OfficialTrain_Augmented")
    statrows+=describe(fva,yva,"OfficialVal")
    statrows+=describe(fhm,yhm,"Handmade")
    for writer in (41,42,43,44):
        m=whm==writer
        statrows+=describe({k:v[m] for k,v in fhm.items()},yhm[m],"Handmade",writer)
    csvwrite(out/"geometry_distribution_summary.csv",statrows)
    center,scale,refrows=reference_stats(ftr,ytr)
    csvwrite(out/"official_class_references.csv",refrows)

    # Source images are 96,000 Training data (76,800 model fit + 19,200 Val).
    rawfeature=out/"all_image_geometry.csv.gz"
    write_features(rawfeature,"OfficialTrain",xtr,ytr,
                   np.full(len(ytr),-1),train_idx,training_aug)
    write_features(rawfeature,"OfficialVal",xva,yva,
                   np.full(len(yva),-1),vaidx)
    write_features(rawfeature,"Handmade",xhm,yhm,whm,ih)
    print(f"Saved per-source geometry for {len(ytr)+len(yva)+len(yhm)} images",flush=True)

    # Class-and-writer-matched descriptive comparisons, without asserting
    # independent effective samples for source-related augmented images.
    pairs=[]
    for class_idx,c in enumerate(LABELS):
        for name,mask,stats in [
            ("Handmade",yhm==class_idx,fhm),
            ("Handmade_41",(yhm==class_idx)&(whm==41),fhm),
            ("Handmade_42",(yhm==class_idx)&(whm==42),fhm),
            ("Handmade_43",(yhm==class_idx)&(whm==43),fhm),
            ("Handmade_44",(yhm==class_idx)&(whm==44),fhm),
            ("OfficialVal",yva==class_idx,fva)
        ]:
            trmask=ytr==class_idx
            row={"class":c,"target":name,"reference":"OfficialTrain",
                 "n_reference":int(trmask.sum()),"n_target":int(mask.sum())}
            for key in ("cx","cy","width","height","bbox_cx","bbox_cy"):
                trainvec=ftr[key][trmask]
                targetvec=stats[key][mask]
                trainvec=trainvec[np.isfinite(trainvec)]
                targetvec=targetvec[np.isfinite(targetvec)]
                row[f"{key}_median_difference"]=float(np.median(targetvec)-np.median(trainvec))
                row[f"{key}_wasserstein"]=float(wasserstein_distance(trainvec,targetvec))
            pairs.append(row)
    csvwrite(out/"per_class_geometry_differences.csv",pairs)

    variants={x.name:x for x in VARIANTS}
    focus=select_focus(xva,yva,pva,vaidx,xhm,yhm,phm,whm,ih)
    selected=[variants[name] for name in COMPARISON]
    focusrows,igrows=focus_analysis(focus,model,selected,out)
    summary3=read_stage3_summary()

    # Controlled comparisons use a fixed model and reference distribution.
    # Ground-truth class reference is only for POST-HOC analysis.
    condition_summary=[]
    group_summary=[]
    relation_summary=[]
    stratified_response=[]
    paired_count=0
    with gzip.open(out/"paired_geometry_model_response.csv.gz","wt",newline="",encoding="utf-8") as f:
        fields=[
          "dataset","writer","image_index","true","original_pred","changed_pred",
          "variant","correct_before","correct_after","outcome",
          "old_class_ref_distance","new_class_ref_distance","distance_change",
          "closer_to_official_train_class_geometry",
          "original_cx","new_cx","original_cy","new_cy",
          "original_width","new_width","original_height","new_height",
          "true_margin_before","true_margin_after","true_margin_change",
          "hidden_cosine","hidden_relative_l2","hidden_active_switch_fraction",
        ]
        w=csv.DictWriter(f,fieldnames=fields)
        w.writeheader()
        domains=[
          ("OfficialVal",xva,yva,np.full(len(yva),-1),vaidx,pva,fva),
          ("Handmade",xhm,yhm,whm,ih,phm,fhm)
        ]
        for domain,raw,y,writers,indices,orig_pred,fo in domains:
            z0,_=get_output(model,raw)
            assert np.array_equal(z0.argmax(axis=1),orig_pred)
            m0=margin(z0,y)
            h0=top_hidden(model,raw)
            d0=reference_distance(fo,y,center,scale)
            for v in selected:
                n=len(y)
                pred=np.empty(n,dtype=np.int64)
                margins=np.empty(n,dtype=np.float64)
                hcos=np.empty(n,dtype=np.float64)
                hl2=np.empty(n,dtype=np.float64)
                switches=np.empty(n,dtype=np.float64)
                g={k:np.empty(n,dtype=np.float64)
                   for k in ("cx","cy","width","height")}
                for left in range(0,n,args.batch):
                    right=min(left+args.batch,n)
                    xx=transform(raw[left:right],v)
                    z,_=get_output(model,xx)
                    pred[left:right]=z.argmax(axis=1)
                    margins[left:right]=margin(z,y[left:right])
                    hc=top_hidden(model,xx)
                    hc0,hrel,hsw=hidden_change(h0[left:right],hc)
                    hcos[left:right]=hc0
                    hl2[left:right]=hrel
                    switches[left:right]=hsw
                    gf=metrics(xx,MAIN_THRESHOLD)
                    for k in g:g[k][left:right]=gf[k]
                d1=reference_distance(g,y,center,scale)
                correction=(orig_pred!=y)&(pred==y)
                induction=(orig_pred==y)&(pred!=y)
                stuck=(orig_pred!=y)&(pred!=y)
                outcome=np.where(correction,"corrected",
                          np.where(induction,"induced",
                          np.where(stuck,"still_wrong","still_correct")))
                closer=d1<d0-1e-5
                for i in range(n):
                    w.writerow({
                        "dataset":domain,"writer":int(writers[i]),
                        "image_index":int(indices[i]),"true":LABELS[y[i]],
                        "original_pred":LABELS[orig_pred[i]],"changed_pred":LABELS[pred[i]],
                        "variant":v.name,
                        "correct_before":int(orig_pred[i]==y[i]),
                        "correct_after":int(pred[i]==y[i]),"outcome":outcome[i],
                        "old_class_ref_distance":float(d0[i]),
                        "new_class_ref_distance":float(d1[i]),
                        "distance_change":float(d1[i]-d0[i]),
                        "closer_to_official_train_class_geometry":int(closer[i]),
                        "original_cx":float(fo["cx"][i]),"new_cx":float(g["cx"][i]),
                        "original_cy":float(fo["cy"][i]),"new_cy":float(g["cy"][i]),
                        "original_width":float(fo["width"][i]),"new_width":float(g["width"][i]),
                        "original_height":float(fo["height"][i]),"new_height":float(g["height"][i]),
                        "true_margin_before":float(m0[i]),
                        "true_margin_after":float(margins[i]),
                        "true_margin_change":float(margins[i]-m0[i]),
                        "hidden_cosine":float(hcos[i]),
                        "hidden_relative_l2":float(hl2[i]),
                        "hidden_active_switch_fraction":float(switches[i]),
                    })
                paired_count+=n
                prev=summary3[(domain,v.name)]
                original_acc=float(np.mean(orig_pred==y))
                acc=float(np.mean(pred==y))
                if abs(acc-float(prev["accuracy"]))>1e-9:
                    raise RuntimeError(f"Stage3 accuracy mismatch: {domain} {v.name} {acc} vs {prev['accuracy']}")
                if int(correction.sum())!=int(prev["corrections"]) or int(induction.sum())!=int(prev["induced_errors"]):
                    raise RuntimeError(f"Stage3 correction/induction mismatch for {domain} {v.name}")
                condition_summary.append({
                    "dataset":domain,"variant":v.name,
                    "n":n,"original_accuracy":original_acc,
                    "accuracy":acc,"corrected":int(correction.sum()),
                    "induced":int(induction.sum()),
                    "mean_hidden_relative_l2":float(hl2.mean()),
                    "mean_hidden_cosine":float(hcos.mean()),
                    "mean_hidden_active_switch_fraction":float(switches.mean()),
                    "mean_geometry_distance_change":float(np.nanmean(d1-d0)),
                    "mean_true_logit_margin_change":float(np.mean(margins-m0)),
                    "n_closer_class_geometry":int(closer.sum()),
                    "fraction_closer_class_geometry":float(closer.mean()),
                })
                for side in ("closer","not_closer"):
                    mm=closer if side=="closer" else ~closer
                    for key,subset in [
                         ("ALL",mm),
                         ("original_wrong",mm&(orig_pred!=y)),
                         ("original_correct",mm&(orig_pred==y))
                    ]:
                        m=int(subset.sum())
                        if m==0:continue
                        relation_summary.append({
                            "dataset":domain,"variant":v.name,
                            "distance_direction":side,"original_status":key,
                            "n":m,"corrected":int((correction&subset).sum()),
                            "induced":int((induction&subset).sum()),
                            "correction_fraction":float((correction&subset).sum()/m),
                            "induction_fraction":float((induction&subset).sum()/m),
                            "mean_delta_true_margin":float((margins-m0)[subset].mean()),
                            "mean_hidden_relative_l2":float(hl2[subset].mean()),
                            "mean_ref_distance_change":float(np.nanmean((d1-d0)[subset])),
                        })
                # Separate every writer and true class BEFORE comparing
                # "closer" against outcome; prevents cross-class aggregation
                # from masquerading as within-class association.
                strata = [("ALL_WRITERS",np.ones(n,dtype=bool))]
                if domain=="Handmade":
                    strata += [(str(ww),writers==ww) for ww in (41,42,43,44)]
                for group_id,member in strata:
                    for cl,c in enumerate(LABELS):
                        for base_state,orig_state in [
                            ("original_wrong",orig_pred!=y),
                            ("original_correct",orig_pred==y)
                        ]:
                            for side,flag in [("closer",closer),("not_closer",~closer)]:
                                subset=member & (y==cl) & orig_state & flag
                                count=int(subset.sum())
                                if not count:continue
                                stratified_response.append({
                                    "dataset":domain,"variant":v.name,
                                    "writer":group_id,"true":c,"baseline_status":base_state,
                                    "direction":side,"n":count,
                                    "corrections":int((correction&subset).sum()),
                                    "induced":int((induction&subset).sum()),
                                    "accuracy_after":float((pred[subset]==y[subset]).mean()),
                                    "mean_delta_true_margin":float((margins-m0)[subset].mean()),
                                })
                for group_name,group_mask in [
                  ("digits",y<10),("letters",y>=10),
                  *([(f"writer_{ww}",writers==ww) for ww in (41,42,43,44)]
                    if domain=="Handmade" else [])
                ]:
                    group_summary.append({
                       "dataset":domain,"variant":v.name,"group":group_name,
                       "n":int(group_mask.sum()),"corrected":int((correction&group_mask).sum()),
                       "induced":int((induction&group_mask).sum()),
                       "mean_distance_change":float(np.nanmean((d1-d0)[group_mask])),
                       "mean_hidden_relative_l2":float(hl2[group_mask].mean()),
                    })
                print(f"{domain} {v.name}: accuracy={acc:.5f}, "
                      f"correction={int(correction.sum())}, "
                      f"induced={int(induction.sum())}, "
                      f"hidden_relative_l2={hl2.mean():.4f}",flush=True)
    assert paired_count==21760*len(COMPARISON)
    csvwrite(out/"spatial_response_summary.csv",condition_summary)
    csvwrite(out/"spatial_response_groups.csv",group_summary)
    csvwrite(out/"geometry_proximity_vs_outcome.csv",relation_summary)
    csvwrite(out/"geometry_proximity_by_writer_class.csv",stratified_response)

    # Diagnostic figures: descriptive, with no fitted regression or causal claims.
    fig,axs=plt.subplots(1,2,figsize=(11,4.5),squeeze=False)
    for j,cls in enumerate(("5","B")):
        target=LABELS.index(cls)
        axs[0,j].scatter(ftr["cx"][ytr==target][::15],ftr["cy"][ytr==target][::15],
                   s=5,alpha=.22,label="Official Train sampled")
        axs[0,j].scatter(fhm["cx"][yhm==target],fhm["cy"][yhm==target],
                   s=16,alpha=.55,label="Handmade")
        axs[0,j].scatter([13.5],[13.5],marker="+",s=80,c="black")
        axs[0,j].set_xlim(0,27);axs[0,j].set_ylim(27,0)
        axs[0,j].set_xlabel("Mass centroid X (pixel)")
        axs[0,j].set_ylabel("Mass centroid Y (pixel)")
        axs[0,j].set_title(f"Class {cls} — geometry by domain")
        axs[0,j].legend(fontsize=7)
    fig.tight_layout();fig.savefig(out/"center_scatter.png",dpi=150);plt.close(fig)
    fig,axs=plt.subplots(2,2,figsize=(10.5,7.5),squeeze=False)
    for j,c in enumerate(("5","9","B","F")):
        ax=axs.flat[j];cid=LABELS.index(c)
        ax.hist(ftr["width"][ytr==cid],bins=np.arange(.5,29.5),density=True,alpha=.45,label="Official Train")
        ax.hist(fhm["width"][yhm==cid],bins=np.arange(.5,29.5),density=True,alpha=.45,label="Handmade")
        ax.set_title(f"{c} — Bounding Box Width");ax.set_xlabel("Width (pixels)")
        ax.legend(fontsize=7)
    fig.tight_layout();fig.savefig(out/"bbox_width_distributions.png",dpi=150);plt.close(fig)
    byclass=[r for r in pairs if r["target"]=="Handmade"]
    fig,axs=plt.subplots(2,1,figsize=(11.7,7.2),squeeze=False)
    for i,key in enumerate(("cx","width")):
        axs[i,0].bar(range(16),[r[f"{key}_median_difference"] for r in byclass])
        axs[i,0].set_xticks(range(16),list(LABELS))
        axs[i,0].axhline(0,color="gray",linewidth=.8)
        axs[i,0].set_ylabel(f"Handmade minus Official Train\nmedian {key} (px)")
    fig.tight_layout();fig.savefig(out/"per_class_spatial_difference.png",dpi=150);plt.close(fig)
    fig,axs=plt.subplots(1,2,figsize=(11.2,4.3),squeeze=False)
    for j,dataset in enumerate(("OfficialVal","Handmade")):
        vals=[r for r in condition_summary if r["dataset"]==dataset and r["variant"]!="original"]
        labels=[v["variant"] for v in vals]
        axs[0,j].bar(range(len(vals)),[v["mean_hidden_relative_l2"] for v in vals])
        axs[0,j].set_xticks(range(len(vals)),labels,rotation=45,ha="right",fontsize=8)
        axs[0,j].set_ylabel("Mean relative L2 change in 100 hidden activations")
        axs[0,j].set_title(dataset)
    fig.tight_layout();fig.savefig(out/"hidden_response_by_transform.png",dpi=150);plt.close(fig)

    valid_thres={}
    for domain,geom in [("OfficialTrain",ftr),("OfficialVal",fva),("Handmade",fhm)]:
        valid_thres[domain]={str(t):int(np.sum(geom["valid"] if t==32 else geom[f"valid_t{t}"]))
                             for t in THRESHOLDS}
    meta={
      "study":"EXP유003 Spatial Distribution & MLP Response",
      "stage3_model_path":str(MODEL_PATH.relative_to(ROOT)),
      "stage3_model_sha256":hashlib.file_digest(MODEL_PATH.open("rb"),"sha256").hexdigest(),
      "model_fitted_here":False,"official_test_read":False,"handmade_used_for_fit":False,
      "official_npz_sha256":off_sha,
      "number_of_source_images":int(len(ytr)+len(yva)+len(yhm)),
      "train":int(len(ytr)),"train_original":int((~training_aug.astype(bool)).sum()),
      "train_augmented":int(training_aug.sum()),
      "validation":int(len(yva)),"handmade":int(len(yhm)),
      "valid_feature_measurements":valid_thres,
      "baseline_official_accuracy":float(accuracy_score(yva,pva)),
      "baseline_handmade_accuracy":float(accuracy_score(yhm,phm)),
      "baseline_official_CM_difference":official_l1,
      "baseline_handmade_prediction_difference":handmade_difference,
      "geometry_thresholds":THRESHOLDS,
      "geometry_reference":"per-class official model training median, robust IQR standardization",
      "target_labels_used_only_for_posthoc_analysis":True,
      "paired_conditions":COMPARISON,
      "paired_response_rows":paired_count,
      "focus_samples":len(focus),"focus_response_rows":len(focusrows),
      "focus_ig_maps":len(igrows),
      "software":{"python":platform.python_version(),"sklearn":sklearn.__version__,
                  "numpy":np.__version__,"scipy":scipy.__version__},
      "elapsed_sec":time.monotonic()-started,
    }
    (out/"run_metadata.json").write_text(json.dumps(meta,ensure_ascii=False,indent=2),
                                          encoding="utf-8")

    source_stats={r["dataset"]:r for r in statrows if r["class"]=="ALL"
                  and (r["dataset"]!="Handmade" or r["writer"]==-1)}
    lines=[
      "# EXP유003 Spatial Distribution & MLP Response — Actual execution summary",
      "",
      f"- Frozen Stage 3 MLP reused, no fitting. Official Test NOT read.",
      f"- Official Train {len(ytr):,} (raw {meta['train_original']:,}, augmented {meta['train_augmented']:,}), "
      f"Validation {len(yva):,}, Handmade {len(yhm):,}.",
      f"- Historical predictions verified: CM-L1={official_l1}, Handmade mismatch={handmade_difference}.",
      f"- Fixed geometry threshold={MAIN_THRESHOLD}; robustness thresholds={list(THRESHOLDS)}.",
      f"- Paired input analyses: {paired_count:,} records, same MLP; focus {len(focus)} images, IG maps {len(igrows)}.",
      "",
      "## Measured geometry (all classes together, exploratory only)",
      "| Data | Center X median | Center Y median | Bounding box width median | height median |",
      "|---|---:|---:|---:|---:|",
    ]
    for src in ("OfficialTrain","OfficialVal","Handmade"):
        r=source_stats[src]
        lines.append(f"| {src} | {r['cx_median']:.3f} | {r['cy_median']:.3f} | "
                     f"{r['width_median']:.3f} | {r['height_median']:.3f} |")
    lines.extend([
      "",
      "## Paired same-image response and errors",
      "| Dataset | Change | Accuracy | Corrected | Newly wrong | Hidden relative L2 | Mean class-distance change |",
      "|---|---|---:|---:|---:|---:|---:|",
    ])
    for rec in condition_summary:
        if rec["variant"]!="original":
            lines.append(f"| {rec['dataset']} | {rec['variant']} | {rec['accuracy']:.3%} "
                         f"| {rec['corrected']} | {rec['induced']} "
                         f"| {rec['mean_hidden_relative_l2']:.3f} "
                         f"| {rec['mean_geometry_distance_change']:+.3f} |")
    lines.extend([
      "",
      "## Interpretation restrictions",
      "- All geometry and prediction associations are observational, even when actual perturbations are applied.",
      "- The perturbations simultaneously change many pixel coordinates and sometimes erase/reshape strokes.",
      "- Class-reference geometry uses TRUE labels for audit only; it is not an allowable deployment rule.",
      "- Grouped augmentation source samples are not independent; do not claim significance from raw counts.",
      "- Integrated Gradients compare the SAME true-vs-original-competitor logit objective within each example.",
      "- Zero-image IG baseline makes dark-background attribution zero by construction.",
      "- This experiment did not select a normalizing transformation or improve a trained model.",
      "",
      "See CSV/PNG/NPZ files for exact counts, classes, writers and fixed-model activations.",
    ])
    (out/"SUMMARY.md").write_text("\n".join(lines)+"\n",encoding="utf-8")
    print(f"EXP003 COMPLETE: {paired_count} paired rows; {len(stratified_response)} strata; {len(focusrows)} focus responses, "
          f"{len(igrows)} IG maps",flush=True)


def main():
    parser=argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--out",type=Path,default=Path("spatial_response_results"))
    parser.add_argument("--batch",type=int,default=1024)
    a=parser.parse_args()
    run(a)


if __name__=="__main__":
    main()


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")